# EVA-X-Tiny: three frozen pretrained pipelines
**Validation gate:** the official checkpoint smoke test must pass before any real-data experiment. Architecture-only tests pass with timm 0.9.16 / torch 2.5.1 / torchvision 0.20.1. The README-suggested timm 0.9.0 fails the official forward pass (missing `_pos_embed`). The cloud proxy blocks Hugging Face with HTTP 403. Thus official pretrained loading, 192D pretrained features and full experiments are **not locally validated**. No results are prefilled. A successful Colab execution must establish these properties rather than assume them.

Compare DenseNet Lung Opacity + OLS, ResNet18 ImageNet 512D + Ridge, and EVA-X-Tiny CXR 192D + Ridge. No prior notebook or result is modified. Outputs: `/content/duong-eva-x-results/`.


In [ ]:
# Colab: run once, restart runtime if requested, then continue below.
%pip install torch==2.5.1 torchvision==0.20.1 torchxrayvision==1.4.0 numpy==1.26.4 pandas==2.2.3 scipy==1.14.1 scikit-learn==1.5.2 scikit-image==0.24.0 matplotlib==3.9.2 timm==0.9.16


In [ ]:
# Optional: mount Drive before configuring historical data.
# from google.colab import drive
# drive.mount('/content/drive')
import os,sys,json,platform,importlib.metadata,shutil
from pathlib import Path
from time import perf_counter
from urllib.parse import urlparse
REPO_DIR=Path(os.environ.get('DUONG_REPO_DIR','/content/nus_DSA5206_GROUPPROJECT')).resolve()
DATA_DIR=Path(os.environ.get('DUONG_DATA_DIR','/content/drive/MyDrive/duong-data')).resolve()
IMAGE_DIR=Path(os.environ.get('DUONG_IMAGE_DIR',str(DATA_DIR/'images')))
METADATA_CSV=Path(os.environ.get('DUONG_METADATA_CSV',str(DATA_DIR/'metadata_2020.csv')))
SCORES_CSV=Path(os.environ.get('DUONG_SCORES_CSV',str(DATA_DIR/'covid-severity-scores.csv')))
ASSESSMENT_DIR=Path(os.environ.get('DUONG_CRITICAL_RESULTS_DIR','/content/duong-critical-assessment-results'))
REPRO_DIR=Path(os.environ.get('DUONG_REPRO_RESULTS_DIR','/content/duong-improved-results'))
RESNET_RESULTS_DIR=Path(os.environ.get('DUONG_RESNET_RESULTS_DIR','/content/duong-resnet18-results'))
OUTPUT_DIR=Path('/content/duong-eva-x-results/').resolve()
assert OUTPUT_DIR!=REPO_DIR and REPO_DIR not in OUTPUT_DIR.parents
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(REPO_DIR));os.chdir(OUTPUT_DIR)
INCLUDE_PCA=False # three prespecified primary pipelines only
import torch,numpy as np,pandas as pd
from PIL import Image
from threadpoolctl import threadpool_limits
from IPython.display import display
from critical_methods import sha256_file,fingerprint,exact_splits
from resnet18_methods import (BASELINE,RIDGE,ALPHAS,SCATTER_SEED,validate_heldout_records,
    paired_alignment,compare_models,aligned_features)
from eva_x_methods import (official_source,official_checkpoint,pretrained_smoke,pooled_features,
    evaluation_transform,PREPROCESSING,OFFICIAL_COMMIT,EVA_MODEL,evaluate_eva,verify_prior_identity)
torch.set_num_threads(2)


## Smoke test first: official model, weights and CXR preprocessing
The pinned official `eva_x.py` factory `eva_x_tiny_patch16` is called unchanged. Its permissive loading is audited: **every non-head backbone tensor must match the filtered checkpoint in key, shape and value**; missing backbone tensors stop execution. Missing random classification-head weights are reported and unused.

Use `forward_head(forward_features(x), pre_logits=True)`: mean of patch tokens excluding prefix tokens, followed by the official `fc_norm`; no classification logits or flattened token sequence. Freeze all parameters, eval mode, inference_mode. Repeat the same synthetic grayscale image and require bitwise-equal finite (1,192) output. Record warnings, checkpoint SHA, device and versions.

Preprocessing follows pinned `classification/utils/datasets.py` **non-CheXpert CXR14 evaluation**: official PIL RGB conversion (grayscale is naturally replicated), bicubic shorter-edge resize 256, center crop 224, tensor [0,1], mean=0.49185243/std=0.28509309 per channel. The official CheXpert-Tiny branch instead resizes to 224 before cropping. There is no uniquely specified COVID severity policy; CXR14 is prespecified here, not selected on test outcomes. No ImageNet normalization is substituted.


In [ ]:
assets=OUTPUT_DIR/'eva_assets'
source=official_source(assets)
local_checkpoint=os.environ.get('DUONG_EVA_CHECKPOINT')
# Expected local-file checksum supplied by the user; not locally verified against weights.
expected_local_sha=os.environ.get('DUONG_EVA_CHECKPOINT_SHA256','f82d6fc450a83ac7c4d32fa9f0d6459ea7a53fd08e47fdef0cb12ce35c97174d')
checkpoint,checkpoint_info=official_checkpoint(assets,local_checkpoint,expected_local_sha)
eva,smoke_report=pretrained_smoke(source,checkpoint,checkpoint_info['sha256'])
assert smoke_report['status']=='passed'
(OUTPUT_DIR/'eva_smoke_report.json').write_text(json.dumps(smoke_report,indent=2))
print(json.dumps(smoke_report,indent=2))


## Reuse the exact validated data and earlier baseline workflow
To avoid a second loader implementation, execute the unchanged existing ResNet notebook's data cell and feature/baseline cells in a dedicated namespace, with OUTPUT_DIR redirected to this new comparison's `baseline_revalidation/`. The old notebook is **read**, never saved or changed. Execution uses the old loader, join/mask, ordered manifests, patient IDs, exact_splits, hash tracking and baseline provenance rules.

When previous ResNet source manifests are available, stop on differences in source CSVs, images, ordered names, patient IDs or targets before doing inference. Copy earlier feature cache into the new working directory only as a read-only source; the original workflow itself verifies its fingerprint/checksum and regenerates when needed. Re-run inexpensive baseline fitting for numerical verification. Historical predictions are reused only after data/weight/feature/preprocessing/environment/split checks **and** agreement with recomputed per-image predictions. Summary means alone never supply predictions.


In [ ]:
assert smoke_report['status']=='passed', 'Do not bypass the pretrained compatibility gate'
assert IMAGE_DIR.is_dir() and METADATA_CSV.is_file() and SCORES_CSV.is_file()
prior_notebook_path=REPO_DIR/'ResNet18_Comparison.ipynb'
prior_notebook=json.loads(prior_notebook_path.read_text())
# The selected cells are located by their reviewed entry points, not cell numbers.
def prior_cell(prefix):
    matches=[''.join(c['source']) for c in prior_notebook['cells']
             if c['cell_type']=='code' and ''.join(c['source']).startswith(prefix)]
    assert len(matches)==1, 'Prior notebook structure changed: review before proceeding'
    return matches[0]
work=OUTPUT_DIR/'baseline_revalidation';work.mkdir(exist_ok=True)
baseline_ns=dict(globals());baseline_ns['OUTPUT_DIR']=work
# Identical data loading, score join/mask, historical split/identity audit.
exec(prior_cell('import numpy as np,pandas as pd,torch,torchvision,torchxrayvision as xrv'),baseline_ns)
metadata=baseline_ns['metadata']
assert len(metadata)==94 and metadata.patientid.nunique()==65
current_identity={
 'ordered_filenames':metadata.filename.tolist(),'ordered_patient_ids':metadata.patientid.tolist(),
 'ordered_labels':metadata[['geographic_mean','opacity_mean']].to_dict(orient='list'),
 'metadata_sha256':sha256_file(METADATA_CSV),'scores_sha256':sha256_file(SCORES_CSV),
 'scored_image_sha256':{f:sha256_file(IMAGE_DIR/f) for f in metadata.filename}}
prior_manifest_path=RESNET_RESULTS_DIR/'reproducibility_manifest.json'
previous=json.loads(prior_manifest_path.read_text()) if prior_manifest_path.is_file() else None
if previous is not None:verify_prior_identity(previous,current_identity)
# Reuse separate old feature caches only through the old validation logic.
for name in ['frozen_features.npz','frozen_features_manifest.json']:
    source_cache=RESNET_RESULTS_DIR/name
    if source_cache.is_file() and not (work/name).exists():shutil.copyfile(source_cache,work/name)
exec(prior_cell("torch.hub.set_dir(str(OUTPUT_DIR/'torch-cache'))"),baseline_ns)
exec(prior_cell('start=perf_counter()\nwith threadpool_limits(limits=2):\n    regenerated=generate_baseline'),baseline_ns)


In [ ]:
# Recompute earlier Ridge with exactly the existing code. This also revalidates
# any historical per-seed predictions numerically rather than trusting a CSV alone.
start=perf_counter()
with threadpool_limits(limits=2):
    prior_metrics,regenerated_predictions,prior_timing,prior_inner=compare_models(
        baseline_ns['resnet_features'],metadata,baseline_ns['baseline'],include_pca=False)
baseline_seconds=perf_counter()-start
earlier=regenerated_predictions
baseline_source='regenerated with validated ResNet/DenseNet code (verified feature cache if compatible)'
old_predictions_path=RESNET_RESULTS_DIR/'all_model_heldout_predictions.csv'
if old_predictions_path.is_file():
    old=pd.read_csv(old_predictions_path,dtype={'patientid':metadata.patientid.dtype})
    old=old[old.model.isin([BASELINE,RIDGE])].copy()
    for name in [BASELINE,RIDGE]:validate_heldout_records(old,metadata,'model',name)
    paired_alignment(old)
    now=baseline_ns['manifest']
    provenance_keys=['resnet_weights','resnet_checkpoint_sha256','resnet_preprocessing','grayscale_policy',
       'checkpoint_sha256','pathologies','packages','outer_split_config','inner_cv','inner_score',
       'ridge_alphas','standardization','ridge_solver','helper_sha256','critical_methods_sha256']
    provenance_match=previous is not None and all(previous.get(k)==now.get(k) for k in provenance_keys)
    if provenance_match:
        keys=['model','target','seed','row_index']
        joined=old.merge(regenerated_predictions,on=keys,validate='one_to_one',suffixes=('_cache','_new'))
        assert len(joined)==len(old)==len(regenerated_predictions)
        np.testing.assert_allclose(joined.y_pred_cache,joined.y_pred_new,rtol=1e-5,atol=1e-5)
        earlier=old
        baseline_source='historical CSV, identity/provenance/splits and numerical predictions revalidated'
    else:print('Earlier feature/model provenance incomplete or different; regenerated baselines used')
print(baseline_source)
earlier.to_csv(OUTPUT_DIR/'verified_earlier_predictions.csv',index=False)


## Dedicated EVA feature cache
Never use the ResNet cache for EVA. The integrity manifest includes ordered filenames/patient IDs/labels, input image and annotation hashes, checkpoint and source hashes, exact transform, pooling, dependency versions and helper hashes. Fingerprint or payload checksum mismatch regenerates EVA features. File names alone are insufficient. No gradients or backbone training are enabled.


In [ ]:
versions={p:importlib.metadata.version(p) for p in ['torch','torchvision','timm','numpy','pandas','scipy','scikit-learn','scikit-image','matplotlib','torchxrayvision']}
manifest={**current_identity,'packages':versions,'python':platform.python_version(),
 'eva_source_commit':OFFICIAL_COMMIT,'eva_source_sha256':sha256_file(source),
 'eva_checkpoint':checkpoint_info,'preprocessing':PREPROCESSING,'pooling':smoke_report['loading']['pooling'],
 'feature_dimension':192,'backbone_frozen':True,'device':'cpu','outer_seeds':list(range(50)),
 'outer_splits':'critical_methods.exact_splits','inner_cv':baseline_ns['manifest']['inner_cv'],
 'ridge_alphas':list(ALPHAS),'inner_score':'negative MSE','scaler':'fold-local StandardScaler',
 'baseline_source':baseline_source,'baseline_manifest':baseline_ns['manifest'],
 'previous_predictions_sha256':sha256_file(old_predictions_path) if old_predictions_path.is_file() else None,
 'shared_notebook_sha256':sha256_file(prior_notebook_path),
 'eva_helper_sha256':sha256_file(REPO_DIR/'eva_x_methods.py'),
 'resnet_helper_sha256':sha256_file(REPO_DIR/'resnet18_methods.py'),
 'critical_helper_sha256':sha256_file(REPO_DIR/'critical_methods.py'),'scatter_seed':SCATTER_SEED}
cache_fingerprint=fingerprint(manifest)
cache=OUTPUT_DIR/'eva_x_features.npz';integrity_path=OUTPUT_DIR/'feature_integrity_manifest.json'
reused=False;extract_seconds=None
if cache.is_file() and integrity_path.is_file():
    integrity=json.loads(integrity_path.read_text())
    if integrity.get('fingerprint')==cache_fingerprint and integrity.get('payload_sha256')==sha256_file(cache):
        with np.load(cache,allow_pickle=False) as payload:eva_features=payload['features'].copy()
        reused=True
if not reused:
    tensors=[];start=perf_counter();transform=evaluation_transform()
    for offset in range(0,len(metadata),8):
        batch=[]
        for name in metadata.filename.iloc[offset:offset+8]:
            with Image.open(IMAGE_DIR/name) as image:batch.append(transform(image.convert('RGB')))
        tensors.append(pooled_features(eva,torch.stack(batch)).cpu().numpy())
    eva_features=np.concatenate(tensors);extract_seconds=perf_counter()-start
aligned_features(eva_features,metadata.filename,metadata,192)
assert eva_features.shape==(94,192)
np.savez(cache,features=eva_features)
integrity_path.write_text(json.dumps({'fingerprint':cache_fingerprint,'payload_sha256':sha256_file(cache),
 'feature_provenance':manifest},indent=2))
metadata.to_csv(OUTPUT_DIR/'ordered_sample_manifest.csv',index=False)
print('Feature cache hit:',reused,'shape:',eva_features.shape)


In [ ]:
start=perf_counter()
with threadpool_limits(limits=2):
    by_seed,predictions,timings,audit=evaluate_eva(eva_features,metadata,earlier)
evaluation_seconds=perf_counter()-start
assert len(by_seed)==300 and len(audit)==50
by_seed.to_csv(OUTPUT_DIR/'three_model_comparison_by_seed.csv',index=False)
summary=by_seed.groupby(['target','model'])[['MAE','MSE','R2','Pearson']].agg(['mean','std','count'])
summary.to_csv(OUTPUT_DIR/'three_model_comparison_summary.csv');display(summary)
formatted=pd.DataFrame(index=summary.index)
for metric in ['MAE','MSE','R2','Pearson']:
    formatted[metric]=[f'{mean:.3f} ± {std:.3f}' if np.isfinite(mean) else 'N/A'
        for mean,std in zip(summary[(metric,'mean')],summary[(metric,'std')])]
display(formatted)
predictions[predictions.model==EVA_MODEL].to_csv(OUTPUT_DIR/'eva_x_heldout_predictions.csv',index=False)
predictions.to_csv(OUTPUT_DIR/'three_model_heldout_predictions.csv',index=False)
paired=by_seed.pivot(index=['target','seed'],columns='model',values='MAE')
paired['EVA-X minus DenseNet']=paired[EVA_MODEL]-paired[BASELINE]
paired['EVA-X minus ResNet18']=paired[EVA_MODEL]-paired[RIDGE]
paired[['EVA-X minus DenseNet','EVA-X minus ResNet18']].to_csv(OUTPUT_DIR/'paired_eva_x_differences.csv')
(OUTPUT_DIR/'patient_split_audit.json').write_text(json.dumps(audit,indent=2,default=lambda v:v.item()))
(OUTPUT_DIR/'reproducibility_manifest.json').write_text(json.dumps(manifest,indent=2))
timings.to_csv(OUTPUT_DIR/'eva_regression_timing.csv',index=False)
(OUTPUT_DIR/'runtime_measurements.json').write_text(json.dumps({'eva_feature_cache_hit':reused,
 'eva_extract_seconds_including_decode_preprocess':extract_seconds,
 'earlier_baseline_search_seconds':baseline_seconds,'eva_evaluation_seconds':evaluation_seconds,
 'earlier_pipeline_resources':baseline_ns['resources']},indent=2))
# User-provided historical means: diagnostics only, never a source of predictions.
reference={('Geographic Extent',BASELINE):(1.160550,.604102),('Geographic Extent',RIDGE):(1.545298,.316714),
 ('Opacity',BASELINE):(.787543,.592061),('Opacity',RIDGE):(.981873,.288485)}
reference_checks=[]
for key,(mae,r2) in reference.items():
    reference_checks.append({'target':key[0],'model':key[1],
        'observed_mean_MAE':summary.loc[key,('MAE','mean')],'reference_mean_MAE':mae,
        'observed_mean_R2':summary.loc[key,('R2','mean')],'reference_mean_R2':r2})
display(pd.DataFrame(reference_checks))
print('Reference deviations require explanation, never changes to seeds or test-tuned settings.')


In [ ]:
import matplotlib.pyplot as plt
models=[BASELINE,RIDGE,EVA_MODEL]
fig,axes=plt.subplots(2,5,figsize=(25,10))
for i,target in enumerate(['Geographic Extent','Opacity']):
    part=by_seed[by_seed.target==target]
    for j,metric in enumerate(['MAE','MSE','R2','Pearson']):
        axes[i,j].boxplot([part[part.model==m][metric].dropna() for m in models],labels=models)
        axes[i,j].tick_params(axis='x',rotation=45);axes[i,j].set_title(target+' '+metric)
    diff=paired.loc[target]
    for column in ['EVA-X minus DenseNet','EVA-X minus ResNet18']:
        axes[i,4].plot(diff.index,diff[column],label=column,alpha=.8)
    axes[i,4].axhline(0,linestyle='--',color='black');axes[i,4].set_title('Paired MAE differences')
    axes[i,4].set_xlabel('Seed');axes[i,4].legend(fontsize=8)
fig.suptitle('Descriptive comparisons: dependent repeated patient splits')
fig.tight_layout();fig.savefig(OUTPUT_DIR/'three_model_comparison.png',dpi=180);plt.show()
fig,axes=plt.subplots(2,3,figsize=(18,10))
for i,(target,score_max) in enumerate([('Geographic Extent',8),('Opacity',6)]):
    p=predictions[(predictions.target==target)&(predictions.seed==SCATTER_SEED)]
    lo=min(-.5,float(p.y_pred.min())-.2);hi=max(score_max+.5,float(p.y_pred.max())+.2)
    for j,model in enumerate(models):
        q=p[p.model==model];axes[i,j].scatter(q.y_true,q.y_pred,alpha=.7)
        axes[i,j].plot([lo,hi],[lo,hi],'--',color='gray')
        axes[i,j].set(xlim=(lo,hi),ylim=(lo,hi),xlabel='Expert mean',ylabel='Held-out prediction',
                      title=f'{target}: {model} — seed {SCATTER_SEED}')
fig.tight_layout();fig.savefig(OUTPUT_DIR/'eva_x_prediction_scatter.png',dpi=180);plt.show()


## Interpretation boundaries
This is a comparison of **complete frozen pretrained pipelines**, not architecture alone. EVA-X and DenseNet have CXR pretraining; ResNet18 uses ImageNet. Input preprocessing, pretraining objectives/populations, 192D versus 512D representations and single-logit versus regularized multivariate regression differ simultaneously. A new internal-performance difference would not isolate any one cause.

Only 65 labelled patients (94 images) support inference. Frozen transfer reduces supervised parameters but does not guarantee robustness. Expert score uncertainty and repeated patient observations limit precision. The 50 patient splits overlap and are dependent; report sample SD and descriptive paired differences, not naive independent-replication significance tests. Negative EVA-X minus baseline MAE favours EVA-X descriptively only. No test-set-directed alpha, crop, pooling or seed tuning is allowed.

**Potential pretraining overlap:** neither repository provenance nor new hashes establish disjointness between the Merged520k pretraining images, the CXR-pretrained DenseNet sources and this historical COVID cohort. Absence of patient overlap across our supervised train/test partitions does not eliminate upstream pretraining overlap. This remains unverified and requires upstream source manifests/deduplication. No external validation, clinical readiness or causal interpretation is established.

**Status:** the checkpoint smoke report must say passed in the current run before these experiments execute. Local cloud checkpoint access was blocked, so full pretrained compatibility and real-data results are unexecuted. No historical summary values are used to populate experiment outputs. Cache hits are reported and do not masquerade as fresh extraction timings.
